In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from torch.utils.data import random_split, DataLoader
from src.configs import SEED, BATCH_SIZE
from src.dataset import ImageDataset
from src.loss_function import Loss
from src.model import Model
from pathlib import Path
import torch

In [23]:
CLASS_TO_IDX = {
    "aeroplane": 0,
    "bicycle": 1,
    "bird": 2,
    "boat": 3,
    "bottle": 4,
    "bus": 5,
    "car": 6,
    "cat": 7,
    "chair": 8,
    "cow": 9,
    "diningtable": 10,
    "dog": 11,
    "horse": 12,
    "motorbike": 13,
    "person": 14,
    "pottedplant": 15,
    "sheep": 16,
    "sofa": 17,
    "train": 18,
    "tvmonitor": 19,
}

IDX_TO_CLASS = {
    idx: class_
    for class_, idx in CLASS_TO_IDX.items()
}

In [3]:
from torchvision.transforms import v2
import torch

# The mean and standard deviations across each channel for the normalized pixels
# of every single image in the "trainval" dataset
MEANS = (0.485, 0.456, 0.406)
STDS = (0.229, 0.224, 0.225)

trainval_transforms = v2.Compose([
    v2.Normalize(mean=MEANS, std=STDS)
])

In [4]:
annot_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annot_file_trainval, img_dir_trainval,
                                transform=trainval_transforms)

generator_ = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(trainval_dataset, [0.0009, 0.9991]
                                          ,generator=generator_)

len(train_dataset)

5

In [5]:
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                     num_workers=4, pin_memory=True, persistent_workers=True)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
model = Model().to(device, non_blocking=True)
loss_fn = Loss().to(device, non_blocking=True)

optimizer = torch.optim.Adam([
    {"params": model.backbone.parameters(), "lr": 1e-4},
    {"params": model.detector_head.parameters()}
    ], lr=1e-4)

In [18]:
for epoch in range(50):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_dl:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model(X_batch)
        loss = loss_fn(preds, y_batch)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(epoch, epoch_loss / len(train_dl))

9 1.415848970413208
19 1.038425326347351
29 0.773827314376831
39 0.6453985571861267
49 0.5404921174049377


In [19]:
from src.inference_functions import compute_eval_stats

train_mAP = compute_eval_stats(model, train_dl, device, test=True)
train_mAP

{'aeroplane': [], 'bicycle': [(tensor(0.1433, device='cuda:0'), False), (tensor(0.1295, device='cuda:0'), False), (tensor(0.1097, device='cuda:0'), False), (tensor(0.1091, device='cuda:0'), False), (tensor(0.0900, device='cuda:0'), False), (tensor(0.0895, device='cuda:0'), False), (tensor(0.1345, device='cuda:0'), False), (tensor(0.1108, device='cuda:0'), False), (tensor(0.1057, device='cuda:0'), False), (tensor(0.0783, device='cuda:0'), False), (tensor(0.0738, device='cuda:0'), False), (tensor(0.0636, device='cuda:0'), False), (tensor(0.1562, device='cuda:0'), False), (tensor(0.1530, device='cuda:0'), False), (tensor(0.1436, device='cuda:0'), False), (tensor(0.1178, device='cuda:0'), False), (tensor(0.1130, device='cuda:0'), False), (tensor(0.0991, device='cuda:0'), False), (tensor(0.0961, device='cuda:0'), False), (tensor(0.0930, device='cuda:0'), False), (tensor(0.0832, device='cuda:0'), False), (tensor(0.0752, device='cuda:0'), False), (tensor(0.0740, device='cuda:0'), False), (ten

(1.0,
 {'bus': 1.0,
  'cat': 1.0,
  'cow': 1.0,
  'person': 1.0,
  'train': 1.0,
  'tvmonitor': 1.0},
 {'aeroplane': ([], []),
  'bicycle': ([], []),
  'bird': ([], []),
  'boat': ([], []),
  'bottle': ([], []),
  'bus': ([1.0,
    1.0,
    1.0,
    0.75,
    0.6,
    0.5,
    0.42857142857142855,
    0.375,
    0.3333333333333333,
    0.3,
    0.2727272727272727,
    0.25,
    0.23076923076923078,
    0.21428571428571427,
    0.2,
    0.1875,
    0.17647058823529413,
    0.16666666666666666,
    0.15789473684210525,
    0.15,
    0.14285714285714285,
    0.13636363636363635,
    0.13043478260869565,
    0.125],
   [0.3333333333333333,
    0.6666666666666666,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0]),
  'car': ([], []),
  'cat': ([1.0,
    0.5,
    0.3333333333333333,
    0.25,
    0.2,
    0.16666666666666666,
    0.14285714285714

In [20]:
ap_by_class = train_mAP[1]
ap_by_class

{'bus': 1.0,
 'cat': 1.0,
 'cow': 1.0,
 'person': 1.0,
 'train': 1.0,
 'tvmonitor': 1.0}

In [11]:
X_batch, y_batch = next(iter(train_dl))
X_batch = X_batch.to(device, non_blocking=True)

X_batch.shape, y_batch.shape

(torch.Size([5, 3, 224, 224]), torch.Size([5, 7, 7, 25]))

In [12]:
preds = model(X_batch)
preds.shape

torch.Size([5, 7, 7, 25])

In [21]:
item_row_col = []
classes = []

for item in range(5):
    for i in range(7):
        for j in range(7):
            if y_batch[item][i][j][-1] == 1:
                item_row_col.append((item, i, j))
                classes.append(y_batch[item][i][j][:20].argmax())

In [29]:
item_row_col, classes

([(0, 4, 0),
  (0, 4, 3),
  (0, 5, 5),
  (1, 3, 3),
  (2, 3, 4),
  (3, 0, 0),
  (3, 0, 1),
  (3, 0, 4),
  (3, 1, 2),
  (3, 1, 3),
  (3, 1, 4),
  (3, 1, 6),
  (4, 4, 5)],
 [tensor(14, device='cuda:0'),
  tensor(14, device='cuda:0'),
  tensor(19, device='cuda:0'),
  tensor(7, device='cuda:0'),
  tensor(18, device='cuda:0'),
  tensor(14, device='cuda:0'),
  tensor(5, device='cuda:0'),
  tensor(5, device='cuda:0'),
  tensor(14, device='cuda:0'),
  tensor(14, device='cuda:0'),
  tensor(14, device='cuda:0'),
  tensor(5, device='cuda:0'),
  tensor(9, device='cuda:0')])

In [30]:
IDX_TO_CLASS[14], IDX_TO_CLASS[19], IDX_TO_CLASS[7], IDX_TO_CLASS[18], IDX_TO_CLASS[5], IDX_TO_CLASS[9]

('person', 'tvmonitor', 'cat', 'train', 'bus', 'cow')

In [15]:
preds[1][0][0], y_batch[1][0][0]

(tensor([-1.6715,  1.7550,  1.0485, -0.1948,  0.4834, -0.0512,  2.0428, -0.8215,
         -1.7722,  1.5163, -0.3327, -0.5165,  2.1831, -0.5520, -0.1888,  1.8620,
         -0.5990,  1.6018,  0.5038, -1.8505,  0.5776,  0.4191,  0.2411,  0.1287,
          0.0784], device='cuda:0', grad_fn=<SelectBackward0>),
 tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0.]))

In [16]:
target_class = y_batch[1][0][0][:20].argmax()
pred_class = preds[1][0][0][:20].argmax()

target_class, pred_class, preds[1][0][0][target_class]

(tensor(0),
 tensor(12, device='cuda:0'),
 tensor(-1.6715, device='cuda:0', grad_fn=<SelectBackward0>))

In [17]:
from src.utilities import IoU, convert_xywh_coordinates

bbox_1 = preds[1][0][0][20:24]
bbox_2 = y_batch[1][0][0][20:24]

bbox_1 = convert_xywh_coordinates(bbox_1, 1, 0, draw=False)
bbox_2 = convert_xywh_coordinates(bbox_2, 1, 0, draw=False)

IoU(bbox_1, bbox_2)

tensor(0., device='cuda:0', grad_fn=<DivBackward0>)